In [1]:
import pandas as pd

In [3]:
df=pd.read_csv("/home/user/Downloads/CAPA-RS-CG/capa_dataset.csv")
df

,incident_id,incident_description,root_cause,corrective_action_type,preventive_action_type,severity_level
0,INC-2019-0001,Relative humidity in manufacturing suite Suite...,Environmental Control Failure,Root cause investigation and CAPA report,Increase monitoring frequency,High
1,INC-2019-0002,Change control CC-6925 closed without completi...,Documentation Error,Document revision and re-issuance,Revise SOP with enhanced controls,Medium
2,INC-2019-0003,Operator OP-8573 set drying temperature to 76....,Human Error,Root cause investigation and CAPA report,Deploy continuous training programme,High
3,INC-2019-0004,pH meter FP-864 gave inconsistent readings dur...,Equipment Failure,Deep cleaning and sanitisation,Implement preventive maintenance schedule,Medium
4,INC-2019-0005,Balance CP-991 failed calibration check before...,Equipment Failure,Process parameter adjustment,Increase monitoring frequency,Low
...,...,...,...,...,...,...
2195,INC-2024-2196,Certificate of Analysis for Microcrystalline C...,Material / Supplier Defect,Document revision and re-issuance,Enhance supplier qualification process,Medium
2196,INC-2024-2197,ERP system LIMS Nexus generated duplicate batc...,Software / System Failure,Product recall and destruction,Add in-process checks at critical steps,High
2197,INC-2024-2198,Cleanroom Room D104 temperature exceeded upper...,Environmental Control Failure,Product recall and destruction,Install automated alarm and alert system,Low
2198,INC-2024-2199,pH meter TBL-338 gave inconsistent readings du...,Equipment Failure,Equipment repair and recalibration,Install automated alarm and alert system,Medium


In [4]:
df.head(1)

,incident_id,incident_description,root_cause,corrective_action_type,preventive_action_type,severity_level
0,INC-2019-0001,Relative humidity in manufacturing suite Suite...,Environmental Control Failure,Root cause investigation and CAPA report,Increase monitoring frequency,High


In [ ]:
pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.7 MB/s eta 0:00:00


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = text.replace("temp", "temperature")
    text = text.replace("sop", "standard operating procedure")

    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
df['cleaned_incident'] = df['incident_description'].apply(clean_text)

In [ ]:
incident_texts = df['cleaned_incident'].tolist()

embeddings = model.encode(incident_texts)

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [ ]:
index.add(np.array(embeddings).astype('float32'))

In [ ]:
faiss.write_index(index, "capa_index.faiss")

In [ ]:
np.save("capa_embeddings.npy", embeddings)

In [ ]:
new_incident = "Temperature deviation observed during vaccine transport"
#new_incident="Electronic signature system"

In [ ]:
query_embedding = model.encode([new_incident])

In [ ]:
k = 3

distances, indices = index.search(
    np.array(query_embedding).astype('float32'),
    k
)

In [ ]:
for i in indices[0]:
    print(df.iloc[i]['incident_description'])

Electronic signature system ChemStation allowed batch E3814C record approval without required second authorised signatory. Workflow configuration error identified.
Electronic signature system QMS Pro v3.2 allowed batch A1624C record approval without required second authorised signatory. Workflow configuration error identified.
Electronic signature system QMS Pro v3.2 allowed batch D9514B record approval without required second authorised signatory. Workflow configuration error identified.


In [ ]:
for i in indices[0]:
    print("Incident:")
    print(df.iloc[i]['incident_description'])

    print("Root Cause:")
    print(df.iloc[i]['root_cause'])

    print("Corrective Action:")
    print(df.iloc[i]['corrective_action_type'])

    print("Preventive Action:")
    print(df.iloc[i]['preventive_action_type'])

    print("Severity:")
    print(df.iloc[i]['severity_level'])

    print("-" * 50)

Incident:
Electronic signature system ChemStation allowed batch E3814C record approval without required second authorised signatory. Workflow configuration error identified.
Root Cause:
Software / System Failure
Corrective Action:
Document revision and re-issuance
Preventive Action:
Upgrade software with validation
Severity:
High
--------------------------------------------------
Incident:
Electronic signature system QMS Pro v3.2 allowed batch A1624C record approval without required second authorised signatory. Workflow configuration error identified.
Root Cause:
Software / System Failure
Corrective Action:
Root cause investigation and CAPA report
Preventive Action:
Deploy continuous training programme
Severity:
High
--------------------------------------------------
Incident:
Electronic signature system QMS Pro v3.2 allowed batch D9514B record approval without required second authorised signatory. Workflow configuration error identified.
Root Cause:
Software / System Failure
Correctiv

In [ ]:
df.to_csv('cleaned_dataset.csv', index=False)

In [ ]:
df

,incident_id,incident_description,root_cause,corrective_action_type,preventive_action_type,severity_level,cleaned_incident
0,INC-2019-0001,Relative humidity in manufacturing suite Suite...,Environmental Control Failure,Root cause investigation and CAPA report,Increase monitoring frequency,High,relative humidity in manufacturing suite suite...
1,INC-2019-0002,Change control CC-6925 closed without completi...,Documentation Error,Document revision and re-issuance,Revise SOP with enhanced controls,Medium,change control cc 6925 closed without completi...
2,INC-2019-0003,Operator OP-8573 set drying temperature to 76....,Human Error,Root cause investigation and CAPA report,Deploy continuous training programme,High,operator op 8573 set drying temperatureerature...
3,INC-2019-0004,pH meter FP-864 gave inconsistent readings dur...,Equipment Failure,Deep cleaning and sanitisation,Implement preventive maintenance schedule,Medium,ph meter fp 864 gave inconsistent readings dur...
4,INC-2019-0005,Balance CP-991 failed calibration check before...,Equipment Failure,Process parameter adjustment,Increase monitoring frequency,Low,balance cp 991 failed calibration check before...
...,...,...,...,...,...,...,...
2195,INC-2024-2196,Certificate of Analysis for Microcrystalline C...,Material / Supplier Defect,Document revision and re-issuance,Enhance supplier qualification process,Medium,certificate of analysis for microcrystalline c...
2196,INC-2024-2197,ERP system LIMS Nexus generated duplicate batc...,Software / System Failure,Product recall and destruction,Add in-process checks at critical steps,High,erp system lims nexus generated duplicate batc...
2197,INC-2024-2198,Cleanroom Room D104 temperature exceeded upper...,Environmental Control Failure,Product recall and destruction,Install automated alarm and alert system,Low,cleanroom room d104 temperatureerature exceede...
2198,INC-2024-2199,pH meter TBL-338 gave inconsistent readings du...,Equipment Failure,Equipment repair and recalibration,Install automated alarm and alert system,Medium,ph meter tbl 338 gave inconsistent readings du...


In [ ]:
from google.colab import files

files.download('cleaned_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>